<a href="https://colab.research.google.com/github/quarcs-lab/geo4cambodia/blob/main/notebooks/districts202_gdppc_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# District-level GDP per capita in Cambodia, 1990–2022: exploratory data analysis

Part of the [GeoDevelopment Observatory of Cambodia](https://quarcs-lab.github.io/geo4cambodia/).

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
import seaborn as sns
import requests

In [2]:
# The geo4cambodia package loads any dataset of the observatory (installed locally by `uv sync`; installed here in Colab).
# Without the package: pd.read_csv("https://raw.githubusercontent.com/quarcs-lab/geo4cambodia/main/data/district/gdppc.csv")
import importlib.util

if importlib.util.find_spec("geo4cambodia") is None:
    !pip install -q geo4cambodia

In [3]:
import geo4cambodia as geo

# District panel: one row per district (geo_id) and year
df = geo.load('gdppc')

# Add district and province names from the GDO unit table
districts = geo.units('district')[['geo_id', 'name', 'parent_id']].rename(columns={'name': 'district'})
provinces = geo.units('province')[['geo_id', 'name']].rename(columns={'geo_id': 'parent_id', 'name': 'province'})
df = df.merge(districts.merge(provinces, on='parent_id').drop(columns='parent_id'), on='geo_id')

In [4]:
df.head()

,geo_id,year,GID_2,y,ln_y,trend6_ln_y,trend100_ln_y,trend400_ln_y,district,province
0,KH-01-01,1990,KHM.1.1_2,676.0,6.516193,6.545359,6.504154,6.428658,Malai,Banteay Meanchey
1,KH-01-01,1991,KHM.1.1_2,711.0,6.566672,6.565005,6.517225,6.462579,Malai,Banteay Meanchey
2,KH-01-01,1992,KHM.1.1_2,651.0,6.478509,6.579986,6.530417,6.496720,Malai,Banteay Meanchey
3,KH-01-01,1993,KHM.1.1_2,1026.0,6.933423,6.585900,6.544344,6.531558,Malai,Banteay Meanchey
4,KH-01-01,1994,KHM.1.1_2,625.0,6.437752,6.562111,6.559102,6.567528,Malai,Banteay Meanchey


## GDP per capita

In [5]:
# Describe 'y' by year
df_mean_y = df.groupby('year')['y'].describe().round(2)

In [6]:
fig = px.line(df_mean_y, x=df_mean_y.index, y=['mean', 'min', 'max'])
fig.update_layout(xaxis_title='Year', yaxis_title='GDP per capita')
fig.show()

In [7]:
# Calculate quantiles by year
yearly_quantiles = df.groupby('year')['y'].quantile([0.1, 0.25, 0.5, 0.75, 0.9]).unstack()

# Melt the DataFrame to long format
yearly_quantiles_long = yearly_quantiles.reset_index().melt(id_vars='year', var_name='quantile', value_name='y_value')

fig = px.line(yearly_quantiles_long, x='year', y='y_value', color='quantile',
              labels={'quantile': 'Percentile', 'y_value': 'y'},
              title='Time Series of Quantiles of y by Year')

fig.update_layout(xaxis_title='Year', yaxis_title='y')

fig.show()

In [8]:
fig = px.box(df, x='year', y='y', hover_name= 'district', hover_data = ['province'],)
fig.update_layout(xaxis_title='Year', yaxis_title='GDP per capita')
fig.show()

In [9]:
px.strip(df,
         x = 'y',
         y = 'province',
         range_x= [400, 12700],
         color = 'province',
         hover_name= 'district',
         hover_data = ['province'],
         animation_frame= 'year')

## Trend of GDP per capita

In [10]:
# Describe 'trend400_ln_y' by year
df_mean_trend400_ln_y = df.groupby('year')['trend400_ln_y'].describe().round(2)

In [11]:
fig = px.line(df_mean_trend400_ln_y, x=df_mean_trend400_ln_y.index, y=['mean', 'min', 'max'])
fig.update_layout(xaxis_title='Year', yaxis_title='Trend of log GDP per capita')
fig.show()

In [12]:
# Calculate quantiles by year
yearly_quantiles = df.groupby('year')['trend400_ln_y'].quantile([0.1, 0.25, 0.5, 0.75, 0.9]).unstack()

# Melt the DataFrame to long format
yearly_quantiles_long = yearly_quantiles.reset_index().melt(id_vars='year', var_name='quantile', value_name='trend400_ln_y_value')

fig = px.line(yearly_quantiles_long, x='year', y='trend400_ln_y_value', color='quantile',
              labels={'quantile': 'Percentile', 'trend400_ln_y_value': 'trend400_ln_y'},
              title='Time Series of Quantiles of trend400_ln_y by Year')

fig.update_layout(xaxis_title='Year', yaxis_title='trend400_ln_y')

fig.show()

In [13]:
fig = px.box(df, x='year', y='trend400_ln_y', hover_name= 'district', hover_data = ['province'],)
fig.update_layout(xaxis_title='Year', yaxis_title='Trend log GDP per capita')
fig.show()

In [14]:
px.strip(df,
         x = 'trend400_ln_y',
         y = 'province',
         range_x= [6.10, 9.5],
         color = 'province',
         hover_name= 'district',
         hover_data = ['province'],
         animation_frame= 'year')